# ShipSense 360 · Data Profiling

## 1. Working Directory

Identify the current working directory to determine the path
to the source CSV.

In [1]:
from pathlib import Path

print("Working folder:", Path.cwd())

Working folder: c:\Users\saths\Desktop\MY Pro\shipsense-360\notebooks


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Raw data folder:", RAW_DIR)
print("Folder exists:", RAW_DIR.exists())

csv_files = sorted(RAW_DIR.rglob("*.csv"))

for file in csv_files:
    print("CSV found:", file.name)

if not csv_files:
    print("No CSV files found in data/raw.")

Raw data folder: c:\Users\saths\Desktop\MY Pro\shipsense-360\data\raw
Folder exists: True
CSV found: DataCoSupplyChainDataset.csv
CSV found: DescriptionDataCoSupplyChain.csv


In [3]:
import pandas as pd

dataset_path = RAW_DIR / "DataCoSupplyChainDataset.csv"

try:
    df = pd.read_csv(dataset_path, encoding="utf-8")
    encoding_used = "utf-8"
except UnicodeDecodeError:
    df = pd.read_csv(dataset_path, encoding="latin1")
    encoding_used = "latin1"

print("Encoding used:", encoding_used)
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print("\nColumn names and data types:")
print(df.dtypes.to_string())

Encoding used: latin1
Rows: 180,519
Columns: 53

Column names and data types:
Type                                 str
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Benefit per order                float64
Sales per customer               float64
Delivery Status                      str
Late_delivery_risk                 int64
Category Id                        int64
Category Name                        str
Customer City                        str
Customer Country                     str
Customer Email                       str
Customer Fname                       str
Customer Id                        int64
Customer Lname                       str
Customer Password                    str
Customer Segment                     str
Customer State                       str
Customer Street                      str
Customer Zipcode                 float64
Department Id                      int64
Department Name                      str
Latitude            

## 3. Record Grain and Missing-Value Checks



In [4]:
# 1. Compare dataset rows with distinct business identifiers
print("RECORD COUNTS")
print(f"Total rows: {len(df):,}")
print(f"Distinct Order Id values: {df['Order Id'].nunique():,}")
print(f"Distinct Order Item Id values: {df['Order Item Id'].nunique():,}")

# 2. Investigate duplicates without removing anything
print("\nDUPLICATE CHECKS")
print(f"Fully duplicated rows: {df.duplicated().sum():,}")

repeated_item_ids = (
    df["Order Item Id"]
    .dropna()
    .duplicated()
    .sum()
)

print(f"Repeated non-null item IDs beyond first occurrence: {repeated_item_ids:,}")

# 3. Show only columns containing missing values
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2)
})

missing_report = (
    missing_report.loc[missing_report["missing_count"] > 0]
    .sort_values("missing_count", ascending=False)
)

print("\nMISSING VALUES")
display(missing_report)

RECORD COUNTS
Total rows: 180,519
Distinct Order Id values: 65,752
Distinct Order Item Id values: 180,519

DUPLICATE CHECKS
Fully duplicated rows: 0
Repeated non-null item IDs beyond first occurrence: 0

MISSING VALUES


,missing_count,missing_percent
Product Description,180519,100.00
Order Zipcode,155679,86.24
Customer Lname,8,0.00
Customer Zipcode,3,0.00


## 4. Order-Level Consistency Checks


In [5]:
# Candidate fields expected to describe an entire order
order_fields = [
    "Order Customer Id",
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Shipping Mode",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delivery Status",
    "Late_delivery_risk",
    "Order Status",
]

# Count distinct values within each order.
# dropna=False also flags a mixture of missing and populated values.
values_per_order = (
    df.groupby("Order Id")[order_fields]
    .nunique(dropna=False)
)

conflict_report = (
    values_per_order.gt(1)
    .sum()
    .rename("orders_with_multiple_values")
    .to_frame()
)

print("ORDER-LEVEL CONSISTENCY")
display(conflict_report)

conflicting_orders = values_per_order.gt(1).any(axis=1)

print(
    "Orders with a conflict in at least one checked field:",
    f"{conflicting_orders.sum():,}"
)

ORDER-LEVEL CONSISTENCY


,orders_with_multiple_values
Order Customer Id,0
order date (DateOrders),0
shipping date (DateOrders),0
Shipping Mode,0
Days for shipping (real),0
Days for shipment (scheduled),0
Delivery Status,0
Late_delivery_risk,0
Order Status,0


Orders with a conflict in at least one checked field: 0


### Observations
- No within-order conflicts were found in any of the nine checked fields.
- Items sharing an Order Id have the same customer ID, order date,
  shipping date, shipping mode, shipping durations, delivery status,
  late-risk indicator, and order status.

## 5. Inspect the Source Data Dictionary


In [6]:
dictionary_path = RAW_DIR / "DescriptionDataCoSupplyChain.csv"

try:
    data_dictionary = pd.read_csv(dictionary_path, encoding="utf-8")
    dictionary_encoding = "utf-8"
except UnicodeDecodeError:
    data_dictionary = pd.read_csv(dictionary_path, encoding="latin1")
    dictionary_encoding = "latin1"

print("Dictionary encoding:", dictionary_encoding)
print("Dictionary shape:", data_dictionary.shape)
print("Dictionary columns:", data_dictionary.columns.tolist())

display(data_dictionary.head(12))

Dictionary encoding: utf-8
Dictionary shape: (52, 2)
Dictionary columns: ['FIELDS', 'DESCRIPTION']


,FIELDS,DESCRIPTION
0,Type,: Type of transaction made
1,Days for shipping (real),: Actual shipping days of the purchased product
2,Days for shipment (scheduled),: Days of scheduled delivery of the purchased...
3,Benefit per order,: Earnings per order placed
4,Sales per customer,: Total sales per customer made per customer
5,Delivery Status,: Delivery status of orders: Advance shipping...
6,Late_delivery_risk,: Categorical variable that indicates if send...
7,Category Id,: Product category code
8,Category Name,: Description of the product category
9,Customer City,: City where the customer made the purchase


## 6. Validate Dictionary Coverage and Read Shipping Definitions

In [7]:
# Normalise names only for comparison; do not rename source columns.
def normalise_field_name(name):
    return " ".join(str(name).split()).casefold()


dataset_names = {
    normalise_field_name(name): name
    for name in df.columns
}

dictionary_keys = data_dictionary["FIELDS"].map(normalise_field_name)
documented_names = set(dictionary_keys)

print("DATASET FIELDS WITHOUT A MATCHING DEFINITION")
unmatched = sorted(set(dataset_names) - documented_names)
print([dataset_names[key] for key in unmatched])

print("\nDICTIONARY FIELDS WITHOUT A MATCHING DATASET COLUMN")
print(
    data_dictionary.loc[
        ~dictionary_keys.isin(dataset_names), "FIELDS"
    ].tolist()
)

print("\nDUPLICATE DICTIONARY FIELD NAMES")
print(
    data_dictionary.loc[
        dictionary_keys.duplicated(keep=False), "FIELDS"
    ].tolist()
)

# Print full descriptions without table-preview truncation.
fields_to_review = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delivery Status",
    "Late_delivery_risk",
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Order Status",
    "Shipping Mode",
]

review_keys = {normalise_field_name(name) for name in fields_to_review}

print("\nCOMPLETE SOURCE DEFINITIONS")
for field, description in data_dictionary.loc[
    dictionary_keys.isin(review_keys), ["FIELDS", "DESCRIPTION"]
].itertuples(index=False, name=None):
    print(f"\n{field}\n{description}")

DATASET FIELDS WITHOUT A MATCHING DEFINITION
['Order Zipcode']

DICTIONARY FIELDS WITHOUT A MATCHING DATASET COLUMN
[]

DUPLICATE DICTIONARY FIELD NAMES
[]

COMPLETE SOURCE DEFINITIONS

Days for shipping (real)     
:  Actual shipping days of the purchased product

Days for shipment (scheduled)
:  Days of scheduled delivery of the purchased product

Delivery Status
:  Delivery status of orders: Advance shipping , Late delivery , Shipping canceled , Shipping on time

Late_delivery_risk           
:  Categorical variable that indicates if sending is late (1), it is not late (0).

order date (DateOrders)
:  Date on which the order is made

Order Status
:  Order Status : COMPLETE , PENDING , CLOSED , PENDING_PAYMENT ,CANCELED , PROCESSING ,SUSPECTED_FRAUD ,ON_HOLD ,PAYMENT_REVIEW

Shipping date (DateOrders)   
:  Exact date and time of shipment

Shipping Mode
:  The following shipping modes are presented : Standard Class , First Class , Second Class , Same Day


### Observations
- All 52 dictionary entries match dataset columns after normalising
  case and whitespace.
- Order Zipcode is the only dataset column without a matching definition.
- No duplicate dictionary field names were found.
- Late_delivery_risk is defined as 1 for late sending and 0 otherwise.
- Delivery Status includes early, late, on-time, and cancelled outcomes.
- The shipping date is defined as the date and time of shipment.

## 7. Compare Shipping Outcome Labels at Order Level

In [8]:
# Select only the identifiers and outcome fields needed for this check.
# Their consistency within each order was verified earlier.
profile_columns = [
    "Order Id",
    "Delivery Status",
    "Late_delivery_risk",
    "Order Status",
]

order_profile = df[profile_columns].drop_duplicates()

# Stop if consolidation unexpectedly leaves multiple records per order.
assert order_profile["Order Id"].is_unique, (
    "Multiple outcome records remain for the same order. Investigate first."
)

print(f"Orders included: {len(order_profile):,}")

print("\nDELIVERY STATUS VS LATE-RISK INDICATOR")
display(
    pd.crosstab(
        order_profile["Delivery Status"],
        order_profile["Late_delivery_risk"],
        margins=True
    )
)

print("\nORDER STATUS VS DELIVERY STATUS")
display(
    pd.crosstab(
        order_profile["Order Status"],
        order_profile["Delivery Status"],
        margins=True
    )
)

Orders included: 65,752

DELIVERY STATUS VS LATE-RISK INDICATOR


Late_delivery_risk,0,1,All
Delivery Status,,,
Advance shipping,15127,0,15127
Late delivery,0,36048,36048
Shipping canceled,2855,0,2855
Shipping on time,11722,0,11722
All,29704,36048,65752



ORDER STATUS VS DELIVERY STATUS


Delivery Status,Advance shipping,Late delivery,Shipping canceled,Shipping on time,All
Order Status,,,,,
CANCELED,0,0,1367,0,1367
CLOSED,1777,4112,0,1360,7249
COMPLETE,5174,12521,0,4021,21716
ON_HOLD,858,2045,0,721,3624
PAYMENT_REVIEW,159,405,0,140,704
PENDING,1797,4215,0,1309,7321
PENDING_PAYMENT,3421,8256,0,2705,14382
PROCESSING,1941,4494,0,1466,7901
SUSPECTED_FRAUD,0,0,1488,0,1488


### Observations
- The analysis includes 65,752 unique orders.
- All 36,048 orders labelled "Late delivery" have Late_delivery_risk = 1.
- Every other delivery-status category has Late_delivery_risk = 0.
- The zero-labelled group includes:
  - 15,127 advance-shipping orders.
  - 11,722 on-time-shipping orders.
  - 2,855 cancelled-shipping orders.
- The 2,855 cancelled shipments consist of:
  - 1,367 orders with Order Status = CANCELED.
  - 1,488 orders with Order Status = SUSPECTED_FRAUD.
- Statuses such as PENDING and PROCESSING also appear alongside
  recorded early, late, and on-time shipping outcomes.

## 8. Validate Shipping Durations Against Outcome Labels

In [9]:
duration_columns = [
    "Order Id",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delivery Status",
]

shipping_profile = df[duration_columns].drop_duplicates().copy()

assert shipping_profile["Order Id"].is_unique, (
    "Multiple shipping records remain for the same order."
)

actual_days = shipping_profile["Days for shipping (real)"]
scheduled_days = shipping_profile["Days for shipment (scheduled)"]

shipping_profile["shipping_day_difference"] = actual_days - scheduled_days
difference = shipping_profile["shipping_day_difference"]

duration_relation = pd.Series(
    "Equal to scheduled", index=shipping_profile.index
)
duration_relation.loc[difference > 0] = "Longer than scheduled"
duration_relation.loc[difference < 0] = "Shorter than scheduled"
duration_relation.loc[difference.isna()] = "Missing duration"

print("DELIVERY STATUS VS SHIPPING-DURATION COMPARISON")
display(
    pd.crosstab(
        shipping_profile["Delivery Status"],
        duration_relation.rename("Duration comparison"),
        margins=True
    )
)

print("\nNEGATIVE-DURATION CHECKS — ORDER COUNTS")
print(f"Negative actual shipping days: {(actual_days < 0).sum():,}")
print(f"Negative scheduled shipping days: {(scheduled_days < 0).sum():,}")

DELIVERY STATUS VS SHIPPING-DURATION COMPARISON


Duration comparison,Equal to scheduled,Longer than scheduled,Shorter than scheduled,All
Delivery Status,,,,
Advance shipping,0,0,15127,15127
Late delivery,0,36048,0,36048
Shipping canceled,545,1650,660,2855
Shipping on time,11722,0,0,11722
All,12267,37698,15787,65752



NEGATIVE-DURATION CHECKS — ORDER COUNTS
Negative actual shipping days: 0
Negative scheduled shipping days: 0


### Observations
- All 15,127 advance-shipping orders have actual days below scheduled days.
- All 36,048 late-labelled orders have actual days above scheduled days.
- All 11,722 on-time-shipping orders have equal actual and scheduled days.
- The 2,855 cancelled shipments contain:
  - 545 with equal durations.
  - 1,650 with actual days above scheduled days.
  - 660 with actual days below scheduled days.
- No negative actual or scheduled shipping durations were found.

## 9. Validate Order and Shipping Timestamp Formats

In [10]:
date_fields = {
    "order date (DateOrders)": "order_datetime",
    "shipping date (DateOrders)": "shipping_datetime",
}

date_profile = (
    df[["Order Id", *date_fields]]
    .drop_duplicates()
    .copy()
)

assert date_profile["Order Id"].is_unique, (
    "Multiple date records remain for the same order."
)

# Format hypothesis to validate against the source strings:
# Example: 1/31/2017 14:30
DATE_FORMAT = "%m/%d/%Y %H:%M"

date_checks = []

for source_column, parsed_column in date_fields.items():
    raw_dates = date_profile[source_column]

    print(f"\n{source_column} — source examples:")
    print(raw_dates.dropna().head(3).tolist())

    parsed_dates = pd.to_datetime(
        raw_dates,
        format=DATE_FORMAT,
        errors="coerce"
    )

    date_profile[parsed_column] = parsed_dates
    failed = raw_dates.notna() & parsed_dates.isna()

    date_checks.append({
        "field": source_column,
        "missing_source": int(raw_dates.isna().sum()),
        "failed_to_parse": int(failed.sum()),
        "earliest_parsed": parsed_dates.min(),
        "latest_parsed": parsed_dates.max(),
    })

    if failed.any():
        print("Examples that failed:", raw_dates.loc[failed].head(3).tolist())

print("\nDATE PARSING REPORT — ORDER LEVEL")
display(pd.DataFrame(date_checks))


order date (DateOrders) — source examples:
['1/31/2018 22:56', '1/13/2018 12:27', '1/13/2018 12:06']

shipping date (DateOrders) — source examples:
['2/3/2018 22:56', '1/18/2018 12:27', '1/17/2018 12:06']

DATE PARSING REPORT — ORDER LEVEL


,field,missing_source,failed_to_parse,earliest_parsed,latest_parsed
0,order date (DateOrders),0,0,2015-01-01,2018-01-31 23:38:00
1,shipping date (DateOrders),0,0,2015-01-03,2018-02-06 22:14:00


### Observations
- Both timestamp fields match the tested month/day/year hour:minute format.
- Neither field contains missing source values or parsing failures.
- Order timestamps range from 2015-01-01 to 2018-01-31.
- Shipping timestamps range from 2015-01-03 to 2018-02-06.

## 10. Reconcile Shipping Timestamps with Recorded Duration

In [11]:
# Match records by Order Id, rather than relying on row position.
duration_validation = date_profile[
    ["Order Id", "order_datetime", "shipping_datetime"]
].merge(
    shipping_profile[
        ["Order Id", "Days for shipping (real)", "Delivery Status"]
    ],
    on="Order Id",
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert duration_validation["_merge"].eq("both").all(), (
    "Some orders are missing from one of the profiling tables."
)

elapsed = (
    duration_validation["shipping_datetime"]
    - duration_validation["order_datetime"]
)

recorded_duration = pd.to_timedelta(
    duration_validation["Days for shipping (real)"],
    unit="D"
)

duration_validation["elapsed_days"] = elapsed.dt.total_seconds() / 86400

missing_dates = elapsed.isna()
shipping_before_order = elapsed < pd.Timedelta(0)
duration_mismatch = elapsed.notna() & elapsed.ne(recorded_duration)

print("TIMESTAMP AND DURATION CHECKS — ORDER COUNTS")
print(f"Orders checked: {len(duration_validation):,}")
print(f"Missing elapsed durations: {missing_dates.sum():,}")
print(f"Shipping before order placement: {shipping_before_order.sum():,}")
print(f"Exact duration mismatches: {duration_mismatch.sum():,}")

if duration_mismatch.any():
    print("\nExample mismatches:")
    display(
        duration_validation.loc[
            duration_mismatch,
            [
                "Order Id",
                "order_datetime",
                "shipping_datetime",
                "Days for shipping (real)",
                "elapsed_days",
                "Delivery Status",
            ]
        ].head(10)
    )

TIMESTAMP AND DURATION CHECKS — ORDER COUNTS
Orders checked: 65,752
Missing elapsed durations: 0
Shipping before order placement: 0
Exact duration mismatches: 3,571

Example mismatches:


,Order Id,order_datetime,shipping_datetime,Days for shipping (real),elapsed_days,Delivery Status
90,114,2015-01-02 15:35:00,2015-01-03 03:35:00,1,0.5,Late delivery
120,149,2015-01-03 03:51:00,2015-01-03 15:51:00,0,0.5,Shipping on time
158,192,2015-01-03 18:54:00,2015-01-04 06:54:00,1,0.5,Late delivery
159,193,2015-01-03 19:15:00,2015-01-04 07:15:00,1,0.5,Late delivery
160,194,2015-01-03 19:36:00,2015-01-04 07:36:00,1,0.5,Late delivery
200,240,2015-01-04 11:43:00,2015-01-04 23:43:00,0,0.5,Shipping on time
231,274,2015-01-04 23:38:00,2015-01-05 11:38:00,1,0.5,Late delivery
232,275,2015-01-04 23:59:00,2015-01-05 11:59:00,1,0.5,Late delivery
233,277,2015-01-05 00:41:00,2015-01-05 12:41:00,0,0.5,Shipping on time
279,330,2015-01-05 19:15:00,2015-01-06 07:15:00,1,0.5,Late delivery


### Observations
- All 65,752 orders have calculable elapsed durations.
- No shipping timestamp occurs before its corresponding order timestamp.
- Recorded shipping days differ from exact elapsed days for 3,571 orders.
- All ten displayed mismatches have an elapsed duration of 0.5 days.
- In these examples, recorded days are 1 when the interval crosses
  midnight and 0 when both timestamps fall on the same date.

## 11. Test Calendar Days Versus Elapsed Days

In [12]:
duration_diagnostics = duration_validation.copy()

# Normalising removes the time of day from temporary datetime values.
# The original timestamp columns remain unchanged.
duration_diagnostics["calendar_days"] = (
    duration_diagnostics["shipping_datetime"].dt.normalize()
    - duration_diagnostics["order_datetime"].dt.normalize()
).dt.days

recorded_days = duration_diagnostics["Days for shipping (real)"]

elapsed_mismatch = duration_diagnostics["elapsed_days"].ne(recorded_days)
calendar_mismatch = duration_diagnostics["calendar_days"].ne(recorded_days)

print("DURATION DEFINITION CHECKS — ORDER COUNTS")
print(f"Orders checked: {len(duration_diagnostics):,}")
print(f"Exact elapsed-day mismatches: {elapsed_mismatch.sum():,}")
print(f"Calendar-day mismatches: {calendar_mismatch.sum():,}")

# Summarise all elapsed-duration mismatches, rather than only examples.
print("\nPATTERNS AMONG EXACT ELAPSED-DAY MISMATCHES")

mismatch_patterns = (
    duration_diagnostics.loc[elapsed_mismatch]
    .groupby(
        ["Days for shipping (real)", "elapsed_days", "calendar_days"],
        dropna=False
    )
    .size()
    .rename("order_count")
    .reset_index()
    .sort_values("order_count", ascending=False)
)

display(mismatch_patterns)

if calendar_mismatch.any():
    print("\nExample orders still inconsistent with calendar days:")
    display(
        duration_diagnostics.loc[
            calendar_mismatch,
            [
                "Order Id",
                "order_datetime",
                "shipping_datetime",
                "Days for shipping (real)",
                "elapsed_days",
                "calendar_days",
            ]
        ].head(10)
    )

DURATION DEFINITION CHECKS — ORDER COUNTS
Orders checked: 65,752
Exact elapsed-day mismatches: 3,571
Calendar-day mismatches: 0

PATTERNS AMONG EXACT ELAPSED-DAY MISMATCHES


,Days for shipping (real),elapsed_days,calendar_days,order_count
0,0,0.5,0,1844
1,1,0.5,1,1727


### Observations
- Recorded actual shipping days match calendar-day differences
  for all 65,752 orders.
- There are zero calendar-day mismatches.
- All 3,571 exact elapsed-day mismatches have a 12-hour duration:
  - 1,844 remain within the same date and record 0 days.
  - 1,727 cross midnight and record 1 day.

## 12. Examine Shipping Modes and Scheduled Durations

In [13]:
mode_columns = [
    "Order Id",
    "Shipping Mode",
    "Days for shipment (scheduled)",
]

mode_profile = df[mode_columns].drop_duplicates()

assert mode_profile["Order Id"].is_unique, (
    "Multiple shipping-mode records remain for the same order."
)

mode_profile = mode_profile.merge(
    duration_diagnostics[["Order Id", "elapsed_days"]],
    on="Order Id",
    how="outer",
    validate="one_to_one",
    indicator="mode_match"
)

assert mode_profile["mode_match"].eq("both").all(), (
    "Some orders are missing from one of the profiling tables."
)

print("SHIPPING MODE VS SCHEDULED DAYS — ORDER COUNTS")
display(
    pd.crosstab(
        mode_profile["Shipping Mode"],
        mode_profile["Days for shipment (scheduled)"],
        margins=True
    )
)

mode_profile["is_12_hour_interval"] = mode_profile["elapsed_days"].eq(0.5)

print("\n12-HOUR INTERVALS BY SHIPPING MODE")
display(
    mode_profile.groupby("Shipping Mode").agg(
        order_count=("Order Id", "size"),
        twelve_hour_orders=("is_12_hour_interval", "sum")
    )
)

SHIPPING MODE VS SCHEDULED DAYS — ORDER COUNTS


Days for shipment (scheduled),0,1,2,4,All
Shipping Mode,,,,,
First Class,0,10079,0,0,10079
Same Day,3571,0,0,0,3571
Second Class,0,0,12778,0,12778
Standard Class,0,0,0,39324,39324
All,3571,10079,12778,39324,65752



12-HOUR INTERVALS BY SHIPPING MODE


,order_count,twelve_hour_orders
Shipping Mode,,
First Class,10079,0
Same Day,3571,3571
Second Class,12778,0
Standard Class,39324,0


### Observations
- Each shipping mode has exactly one scheduled duration in this dataset.
- All Same Day orders have an elapsed duration of 12 hours.
- All previously identified elapsed-day mismatches belong to Same Day orders.

## 13. Inspect Commercial Field Definitions and Distributions

In [14]:
commercial_fields = [
    "Order Item Quantity",
    "Order Item Product Price",
    "Sales",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Total",
    "Benefit per order",
    "Sales per customer",
    "Order Profit Per Order",
    "Order Item Profit Ratio",
]

commercial_keys = {
    normalise_field_name(field)
    for field in commercial_fields
}

print("COMPLETE SOURCE DEFINITIONS")
for field, description in data_dictionary.loc[
    dictionary_keys.isin(commercial_keys),
    ["FIELDS", "DESCRIPTION"]
].itertuples(index=False, name=None):
    print(f"\n{field}\n{description}")

print("\nCOMMERCIAL FIELD DISTRIBUTIONS — ORDER-ITEM LEVEL")
display(
    df[commercial_fields]
    .describe()
    .T
    .round(4)
)

COMPLETE SOURCE DEFINITIONS

Benefit per order
:  Earnings per order placed

Sales per customer
:  Total sales per customer made per customer

Order Item Discount
:  Order item discount value

Order Item Discount Rate     
:  Order item discount percentage

Order Item Product Price     
:  Price of products without discount

Order Item Profit Ratio
:  Order Item Profit Ratio

Order Item Quantity
:  Number of products per order

Sales
:  Value in sales

Order Item Total  
:  Total amount per order

Order Profit Per Order
:  Order Profit Per Order

COMMERCIAL FIELD DISTRIBUTIONS — ORDER-ITEM LEVEL


,count,mean,std,min,25%,50%,75%,max
Order Item Quantity,180519.0,2.1276,1.4535,1.00,1.00,1.00,3.00,5.00
Order Item Product Price,180519.0,141.2325,139.7325,9.99,50.00,59.99,199.99,1999.99
Sales,180519.0,203.7721,132.2731,9.99,119.98,199.92,299.95,1999.99
Order Item Discount,180519.0,20.6647,21.8009,0.00,5.40,14.00,29.99,500.00
Order Item Discount Rate,180519.0,0.1017,0.0704,0.00,0.04,0.10,0.16,0.25
Order Item Total,180519.0,183.1076,120.0437,7.49,104.38,163.99,247.40,1939.99
Benefit per order,180519.0,21.9750,104.4335,-4274.98,7.00,31.52,64.80,911.80
Sales per customer,180519.0,183.1076,120.0437,7.49,104.38,163.99,247.40,1939.99
Order Profit Per Order,180519.0,21.9750,104.4335,-4274.98,7.00,31.52,64.80,911.80
Order Item Profit Ratio,180519.0,0.1206,0.4668,-2.75,0.08,0.27,0.36,0.50


### Observations
- All ten inspected commercial fields have 180,519 non-missing values.
- Quantities range from 1 to 5; inspected prices and sales are positive.
- Discount rates range from 0 to 0.25.
- Benefit per order and Order Profit Per Order include negative values.
- Sales per customer and Order Item Total have identical displayed
  summary statistics.
- Benefit per order and Order Profit Per Order also have identical
  displayed summary statistics.
- Some source definitions are vague or repeat the field name.

## 14. Validate Commercial Arithmetic and Duplicate Measures

In [15]:
import numpy as np

AMOUNT_TOLERANCE = 0.01

# Each entry contains the recorded field and its candidate comparison.
commercial_checks = {
    "Sales = quantity × price": (
        "Sales",
        df["Order Item Quantity"] * df["Order Item Product Price"]
    ),
    "Discount = Sales × discount rate": (
        "Order Item Discount",
        df["Sales"] * df["Order Item Discount Rate"]
    ),
    "Item total = Sales − discount": (
        "Order Item Total",
        df["Sales"] - df["Order Item Discount"]
    ),
    "Sales per customer = item total": (
        "Sales per customer",
        df["Order Item Total"]
    ),
    "Benefit = Order Profit Per Order": (
        "Benefit per order",
        df["Order Profit Per Order"]
    ),
    "Profit = item total × profit ratio": (
        "Order Profit Per Order",
        df["Order Item Total"] * df["Order Item Profit Ratio"]
    ),
}

check_results = []

for relationship, (actual_column, expected) in commercial_checks.items():
    actual = df[actual_column]
    valid = np.isfinite(actual) & np.isfinite(expected)
    absolute_error = (actual - expected).abs()

    within_tolerance = pd.Series(
        np.isclose(
            actual,
            expected,
            atol=AMOUNT_TOLERANCE,
            rtol=0
        ),
        index=df.index
    )

    check_results.append({
        "relationship": relationship,
        "tested_rows": int(valid.sum()),
        "exact_matches": int((valid & actual.eq(expected)).sum()),
        "within_tolerance": int((valid & within_tolerance).sum()),
        "outside_tolerance": int((valid & ~within_tolerance).sum()),
        "untestable_rows": int((~valid).sum()),
        "max_absolute_error": absolute_error.loc[valid].max(),
    })

commercial_validation_report = pd.DataFrame(check_results)

display(commercial_validation_report)

,relationship,tested_rows,exact_matches,within_tolerance,outside_tolerance,untestable_rows,max_absolute_error
0,Sales = quantity × price,180519,126929,180519,0,0,0.000023
1,Discount = Sales × discount rate,180519,14387,170490,10029,0,9.999397
2,Item total = Sales − discount,180519,99554,179295,1224,0,0.010014
3,Sales per customer = item total,180519,180519,180519,0,0,0.000000
4,Benefit = Order Profit Per Order,180519,180519,180519,0,0,0.000000
5,Profit = item total × profit ratio,180519,2726,115129,65390,0,9.299195


## 15. Investigate Financial Discrepancies

In [16]:
rate_checks = [
    ("Discount", "Sales", "Order Item Discount", "Order Item Discount Rate"),
    ("Profit", "Order Item Total", "Order Profit Per Order", "Order Item Profit Ratio"),
]

for label, base_column, amount_column, rate_column in rate_checks:
    diagnostic = pd.DataFrame({
        "Order Item Id": df["Order Item Id"],
        "base_amount": df[base_column],
        "recorded_amount": df[amount_column],
        "reported_rate": df[rate_column],
    })

    # A zero base cannot provide a meaningful implied rate.
    diagnostic["implied_rate"] = (
        diagnostic["recorded_amount"]
        / diagnostic["base_amount"].replace(0, np.nan)
    )

    diagnostic["rate_absolute_gap"] = (
        diagnostic["implied_rate"] - diagnostic["reported_rate"]
    ).abs()

    diagnostic["amount_absolute_error"] = (
        diagnostic["recorded_amount"]
        - diagnostic["base_amount"] * diagnostic["reported_rate"]
    ).abs()

    print(f"\n{label.upper()} — DISCREPANCY DISTRIBUTION")
    display(
        diagnostic[["amount_absolute_error", "rate_absolute_gap"]]
        .describe(percentiles=[0.5, 0.95, 0.99])
        .T
    )

    print(f"{label.upper()} — FIVE LARGEST AMOUNT DIFFERENCES")
    display(diagnostic.nlargest(5, "amount_absolute_error"))

# Rounding here groups residuals for display; source values are unchanged.
net_residual = (
    df["Order Item Total"]
    - (df["Sales"] - df["Order Item Discount"])
)

print("\nITEM-TOTAL ABSOLUTE RESIDUALS — GROUPED TO FOUR DECIMAL PLACES")
display(
    net_residual.abs()
    .round(4)
    .value_counts()
    .rename_axis("absolute_residual")
    .reset_index(name="item_count")
    .sort_values("absolute_residual")
)


DISCOUNT — DISCREPANCY DISTRIBUTION


,count,mean,std,min,50%,95%,99%,max
amount_absolute_error,180519.0,0.058072,0.280104,0.0,0.001498,0.299400,1.4988,9.999397
rate_absolute_gap,180519.0,0.000289,0.001142,0.0,0.000008,0.004978,0.0050,0.005407


DISCOUNT — FIVE LARGEST AMOUNT DIFFERENCES


,Order Item Id,base_amount,recorded_amount,reported_rate,implied_rate,rate_absolute_gap,amount_absolute_error
82117,172140,1999.98999,110.0,0.06,0.055,0.005,9.999397
152,174049,1500.00000,82.5,0.06,0.055,0.005,7.499999
3204,177839,1500.00000,82.5,0.06,0.055,0.005,7.499999
11813,174157,1500.00000,82.5,0.06,0.055,0.005,7.499999
12531,173869,1500.00000,82.5,0.06,0.055,0.005,7.499999



PROFIT — DISCREPANCY DISTRIBUTION


,count,mean,std,min,50%,95%,99%,max
amount_absolute_error,180519.0,0.219372,0.405610,0.0,0.004198,1.052406,1.746008,9.299195
rate_absolute_gap,180519.0,0.001208,0.001737,0.0,0.000029,0.004996,0.005028,0.005505


PROFIT — FIVE LARGEST AMOUNT DIFFERENCES


,Order Item Id,base_amount,recorded_amount,reported_rate,implied_rate,rate_absolute_gap,amount_absolute_error
173436,172129,1859.98999,139.5,0.08,0.075000,0.005000,9.299195
13343,172032,1759.98999,660.0,0.38,0.375002,0.004998,8.796187
59847,171806,1599.98999,520.0,0.33,0.325002,0.004998,7.996717
38434,173929,1500.00000,-337.5,-0.23,-0.225000,0.005000,7.500006
26354,174055,1500.00000,112.5,0.08,0.075000,0.005000,7.499997



ITEM-TOTAL ABSOLUTE RESIDUALS — GROUPED TO FOUR DECIMAL PLACES


,absolute_residual,item_count
0,0.00,175906
1,0.01,4613


### Observations
- The largest discount discrepancies include a reported rate of 0.06
  and an implied rate near 0.055.
- Large profit discrepancies show similar differences between
  published and implied rates, including negative profit.
- Maximum absolute rate gaps are approximately:
  - Discount: 0.005407.
  - Profit: 0.005505.
- Absolute item-total residuals, rounded to four decimal places, are:
  - 0.00 for 175,906 items.
  - 0.01 for 4,613 items.
- These grouped counts describe rounded residuals, not exact equality.

## 16. Investigate Financial Grain Within Orders

In [17]:
financial_fields = [
    "Sales",
    "Order Item Total",
    "Sales per customer",
    "Benefit per order",
    "Order Profit Per Order",
]

order_sizes = df.groupby("Order Id").size()
multi_item_order_ids = order_sizes.index[order_sizes > 1]

financial_values_per_order = (
    df.groupby("Order Id")[financial_fields]
    .nunique(dropna=False)
)

grain_report = (
    financial_values_per_order
    .loc[multi_item_order_ids]
    .gt(1)
    .sum()
    .rename("multi_item_orders_with_varying_values")
    .to_frame()
)

print(f"Total orders: {len(order_sizes):,}")
print(f"Single-item orders: {order_sizes.eq(1).sum():,}")
print(f"Multi-item orders: {len(multi_item_order_ids):,}")

display(grain_report)

# Inspect one complete order with differing recorded profit values.
example_ids = financial_values_per_order.index[
    financial_values_per_order["Order Profit Per Order"] > 1
]

if len(example_ids) > 0:
    example_order_id = example_ids[0]
    print(f"\nExample order with varying profit: {example_order_id}")

    display(
        df.loc[
            df["Order Id"].eq(example_order_id),
            [
                "Order Id",
                "Order Item Id",
                "Order Item Quantity",
                *financial_fields,
            ]
        ].sort_values("Order Item Id")
    )

Total orders: 65,752
Single-item orders: 19,850
Multi-item orders: 45,902


,multi_item_orders_with_varying_values
Sales,45190
Order Item Total,45902
Sales per customer,45902
Benefit per order,45902
Order Profit Per Order,45902



Example order with varying profit: 2


,Order Id,Order Item Id,Order Item Quantity,Sales,Order Item Total,Sales per customer,Benefit per order,Order Profit Per Order
77011,2,2,1,199.990005,193.990005,193.990005,91.180000,91.180000
109322,2,3,5,250.000000,227.500000,227.500000,68.250000,68.250000
87884,2,4,1,129.990005,107.889999,107.889999,36.470001,36.470001


### Observations
- There are 19,850 single-item orders and 45,902 multi-item orders.
- Sales varies within 45,190 multi-item orders.
- Item total and both profit fields vary within all 45,902
  multi-item orders.
- Sales per customer continues to mirror Order Item Total.
- Benefit per order continues to mirror Order Profit Per Order.

## 17. Aggregate Commercial Measures and Reconcile Totals

In [18]:
# Guard the item grain before summing financial values.
assert df["Order Item Id"].notna().all()
assert df["Order Item Id"].is_unique

order_commercial_profile = (
    df.groupby("Order Id", as_index=False)
    .agg(
        item_count=("Order Item Id", "size"),
        total_quantity=("Order Item Quantity", "sum"),
        recorded_line_sales=("Sales", "sum"),
        recorded_discounts=("Order Item Discount", "sum"),
        recorded_net_amount=("Order Item Total", "sum"),
        recorded_item_profit=("Order Profit Per Order", "sum"),
    )
)

assert len(order_commercial_profile) == df["Order Id"].nunique()
assert order_commercial_profile["item_count"].sum() == len(df)

measure_mapping = {
    "total_quantity": "Order Item Quantity",
    "recorded_line_sales": "Sales",
    "recorded_discounts": "Order Item Discount",
    "recorded_net_amount": "Order Item Total",
    "recorded_item_profit": "Order Profit Per Order",
}

reconciliation_rows = []

for measure, source_column in measure_mapping.items():
    source_total = df[source_column].sum()
    aggregated_total = order_commercial_profile[measure].sum()

    reconciliation_rows.append({
        "measure": measure,
        "source_total": source_total,
        "aggregated_total": aggregated_total,
        "difference": aggregated_total - source_total,
        "within_0.01": bool(np.isclose(
            source_total, aggregated_total, atol=0.01, rtol=0
        )),
    })

print(f"Order-level records: {len(order_commercial_profile):,}")
print(
    "Source items represented:",
    f"{order_commercial_profile['item_count'].sum():,}"
)

print("\nSOURCE-TO-AGGREGATE RECONCILIATION")
display(pd.DataFrame(reconciliation_rows))

print("\nEXAMPLE: ORDER 2")
display(
    order_commercial_profile.loc[
        order_commercial_profile["Order Id"].eq(2)
    ]
)

Order-level records: 65,752
Source items represented: 180,519

SOURCE-TO-AGGREGATE RECONCILIATION


,measure,source_total,aggregated_total,difference,within_0.01
0,total_quantity,3.840790e+05,3.840790e+05,0.0,True
1,recorded_line_sales,3.678474e+07,3.678474e+07,0.0,True
2,recorded_discounts,3.730378e+06,3.730378e+06,0.0,True
3,recorded_net_amount,3.305440e+07,3.305440e+07,0.0,True
4,recorded_item_profit,3.966903e+06,3.966903e+06,0.0,True



EXAMPLE: ORDER 2


,Order Id,item_count,total_quantity,recorded_line_sales,recorded_discounts,recorded_net_amount,recorded_item_profit
1,2,3,7,579.980011,50.6,529.380005,195.900002


### Observations
- The commercial profiling table contains 65,752 unique orders.
- Its item counts account for all 180,519 source records.
- Total quantity is 384,079 units.
- Quantity, line sales, discounts, net amounts, and recorded item
  profit reconcile with zero reported difference.
- Order 2 contains:
  - 3 item records and 7 units.
  - Recorded line sales of approximately 579.98.
  - Discounts of approximately 50.60.
  - Recorded net amount of approximately 529.38.
  - Recorded item profit of approximately 195.90.

## 18. Validate Identifier Relationships and Product Hierarchy

In [19]:
identifier_pairs = [
    ("Order Customer Id", "Customer Id"),
    ("Order Item Cardprod Id", "Product Card Id"),
    ("Category Id", "Product Category Id"),
]

identifier_results = []

for left, right in identifier_pairs:
    valid = df[left].notna() & df[right].notna()

    identifier_results.append({
        "comparison": f"{left} = {right}",
        "comparable_rows": int(valid.sum()),
        "missing_either_id": int((~valid).sum()),
        "mismatched_rows": int(
            (valid & df[left].ne(df[right])).sum()
        ),
    })

print("IDENTIFIER AGREEMENT")
display(pd.DataFrame(identifier_results))

# Check whether each identifier has one consistent attribute value.
# The reverse relationship need not be unique:
# multiple products may belong to the same category.
hierarchy_checks = [
    ("Product Card Id", "Product Name"),
    ("Product Card Id", "Product Category Id"),
    ("Category Id", "Category Name"),
    ("Category Id", "Department Id"),
    ("Department Id", "Department Name"),
]

hierarchy_results = []

for identifier, attribute in hierarchy_checks:
    values_per_id = (
        df.groupby(identifier)[attribute]
        .nunique(dropna=False)
    )

    hierarchy_results.append({
        "identifier": identifier,
        "attribute": attribute,
        "distinct_non_null_ids": len(values_per_id),
        "ids_with_multiple_attribute_values": int(
            values_per_id.gt(1).sum()
        ),
        "rows_missing_identifier": int(
            df[identifier].isna().sum()
        ),
        "rows_missing_attribute": int(
            df[attribute].isna().sum()
        ),
    })

print("\nPRODUCT-HIERARCHY CONSISTENCY")
display(pd.DataFrame(hierarchy_results))

IDENTIFIER AGREEMENT


,comparison,comparable_rows,missing_either_id,mismatched_rows
0,Order Customer Id = Customer Id,180519,0,0
1,Order Item Cardprod Id = Product Card Id,180519,0,0
2,Category Id = Product Category Id,180519,0,0



PRODUCT-HIERARCHY CONSISTENCY


,identifier,attribute,distinct_non_null_ids,ids_with_multiple_attribute_values,rows_missing_identifier,rows_missing_attribute
0,Product Card Id,Product Name,118,0,0,0
1,Product Card Id,Product Category Id,118,0,0,0
2,Category Id,Category Name,51,0,0,0
3,Category Id,Department Id,51,0,0,0
4,Department Id,Department Name,11,0,0,0


### Observations
- All 180,519 records have matching customer identifiers:
  Order Customer Id = Customer Id.
- All records have matching product identifiers:
  Order Item Cardprod Id = Product Card Id.
- All records have matching category identifiers:
  Category Id = Product Category Id.
- Each of the 118 product IDs maps to one product name and category.
- Each of the 51 category IDs maps to one category name and department.
- Each of the 11 department IDs maps to one department name.
- No missing identifiers or attributes were found in these checks.

## 19. Validate Order Geography and Categorical Text

In [20]:
geography_fields = [
    "Market",
    "Order Region",
    "Order Country",
    "Order State",
    "Order City",
    "Customer Segment",
]

geography_values_per_order = (
    df.groupby("Order Id")[geography_fields]
    .nunique(dropna=False)
)

print("ORDER GEOGRAPHY AND SEGMENT CONSISTENCY")
display(
    geography_values_per_order.gt(1)
    .sum()
    .rename("orders_with_multiple_values")
    .to_frame()
)

text_fields = geography_fields + [
    "Shipping Mode",
    "Delivery Status",
    "Order Status",
    "Type",
    "Category Name",
    "Department Name",
]

text_results = []

for column in text_fields:
    original = df[column].astype("string")

    # Temporary comparison values; source columns remain unchanged.
    normalised = (
        original.str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.casefold()
    )

    text_results.append({
        "field": column,
        "missing_values": int(original.isna().sum()),
        "blank_strings": int(
            original.str.fullmatch(r"\s*", na=False).sum()
        ),
        "rows_with_outer_whitespace": int(
            original.ne(original.str.strip()).sum()
        ),
        "distinct_original_labels": original.nunique(),
        "distinct_normalised_labels": normalised.nunique(),
    })

print("\nCATEGORICAL TEXT QUALITY")
display(pd.DataFrame(text_results))

ORDER GEOGRAPHY AND SEGMENT CONSISTENCY


,orders_with_multiple_values
Market,0
Order Region,0
Order Country,0
Order State,0
Order City,0
Customer Segment,0



CATEGORICAL TEXT QUALITY


,field,missing_values,blank_strings,rows_with_outer_whitespace,distinct_original_labels,distinct_normalised_labels
0,Market,0,0,0,5,5
1,Order Region,0,0,17925,23,23
2,Order Country,0,0,0,164,164
3,Order State,0,0,0,1089,1089
4,Order City,0,0,0,3597,3597
5,Customer Segment,0,0,0,3,3
6,Shipping Mode,0,0,0,4,4
7,Delivery Status,0,0,0,4,4
8,Order Status,0,0,0,9,9
9,Type,0,0,0,4,4


### Observations
- Market, region, country, state, city, and customer segment are
  consistent within each order.
- None of the 12 inspected text fields contains missing values
  or blank strings.
- Surrounding whitespace occurs in:
  - Order Region: 17,925 item rows.
  - Category Name: 1,475 item rows.
  - Department Name: 362 item rows.
- Normalising case and whitespace does not reduce the distinct-label
  count in any inspected field.
- There are 51 category IDs but only 50 distinct category names.

## 20. Trim Categorical Whitespace and Inspect Shared Category Names

In [21]:
# Clean only the previously inspected categorical fields.
# Keep original spelling, case, accents, and internal whitespace.
categorical_clean = (
    df[text_fields]
    .astype("string")
    .apply(lambda column: column.str.strip())
)

trim_results = []

for column in text_fields:
    original = df[column].astype("string")
    cleaned = categorical_clean[column]

    trim_results.append({
        "field": column,
        "changed_item_rows": int(original.ne(cleaned).sum()),
        "remaining_outer_whitespace": int(
            cleaned.ne(cleaned.str.strip()).sum()
        ),
        "distinct_labels_before": original.nunique(),
        "distinct_labels_after": cleaned.nunique(),
    })

print("WHITESPACE CLEANING AUDIT")
display(pd.DataFrame(trim_results))

# Retain IDs when inspecting names; do not merge categories by name.
category_lookup = pd.DataFrame({
    "Category Id": df["Category Id"],
    "category_name_clean": categorical_clean["Category Name"],
}).drop_duplicates()

ids_per_name = (
    category_lookup.groupby("category_name_clean")["Category Id"]
    .nunique()
)

shared_names = ids_per_name.index[ids_per_name > 1]

print("\nCATEGORY NAMES ASSOCIATED WITH MULTIPLE IDS")
display(
    category_lookup.loc[
        category_lookup["category_name_clean"].isin(shared_names)
    ].sort_values(["category_name_clean", "Category Id"])
)

WHITESPACE CLEANING AUDIT


,field,changed_item_rows,remaining_outer_whitespace,distinct_labels_before,distinct_labels_after
0,Market,0,0,5,5
1,Order Region,17925,0,23,23
2,Order Country,0,0,164,164
3,Order State,0,0,1089,1089
4,Order City,0,0,3597,3597
5,Customer Segment,0,0,3,3
6,Shipping Mode,0,0,4,4
7,Delivery Status,0,0,4,4
8,Order Status,0,0,9,9
9,Type,0,0,4,4



CATEGORY NAMES ASSOCIATED WITH MULTIPLE IDS


,Category Id,category_name_clean
55,13,Electronics
62,37,Electronics


## 21. Build the Initial Order-Level Analytical Table

In [22]:
order_column_mapping = {
    "Order Id": "order_id",
    "Order Customer Id": "customer_id",
    "Days for shipping (real)": "actual_shipping_calendar_days",
    "Days for shipment (scheduled)": "scheduled_shipping_days",
    "Shipping Mode": "shipping_mode",
    "Delivery Status": "delivery_status_source",
    "Late_delivery_risk": "late_delivery_risk_source",
    "Order Status": "order_status_source",
    "Customer Segment": "customer_segment",
    "Market": "market",
    "Order Region": "order_region",
    "Order Country": "order_country",
    "Order State": "order_state",
    "Order City": "order_city",
}

# Select required attributes while preserving the original dataframe.
order_candidates = df[list(order_column_mapping)].copy()

# Bring in the cleaned categorical values using the shared source index.
for column in order_column_mapping:
    if column in categorical_clean.columns:
        order_candidates[column] = categorical_clean[column]

# Consolidate identical order attributes across their item records.
orders_core = (
    order_candidates
    .drop_duplicates()
    .rename(columns=order_column_mapping)
)

assert orders_core["order_id"].notna().all()
assert orders_core["order_id"].is_unique
assert len(orders_core) == df["Order Id"].nunique()

# Attach validated timestamps by business identifier.
orders_core = orders_core.merge(
    date_profile[
        ["Order Id", "order_datetime", "shipping_datetime"]
    ].rename(columns={"Order Id": "order_id"}),
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="timestamp_match"
)

assert orders_core["timestamp_match"].eq("both").all()
assert orders_core[
    ["order_datetime", "shipping_datetime"]
].notna().all().all()

orders_core = orders_core.drop(columns="timestamp_match")

# Keep cancellation separate from the source late-risk indicator.
orders_core["is_shipping_cancelled"] = (
    orders_core["delivery_status_source"].eq("Shipping canceled")
)

orders_core["shipping_days_difference"] = (
    orders_core["actual_shipping_calendar_days"]
    - orders_core["scheduled_shipping_days"]
)

print(f"Order records: {len(orders_core):,}")
print(f"Columns: {orders_core.shape[1]}")
print(f"Duplicate order IDs: {orders_core['order_id'].duplicated().sum():,}")

print("\nORDER TABLE SCHEMA")
display(orders_core.dtypes.astype(str).to_frame("data_type"))

Order records: 65,752
Columns: 18
Duplicate order IDs: 0

ORDER TABLE SCHEMA


,data_type
order_id,int64
customer_id,int64
actual_shipping_calendar_days,int64
scheduled_shipping_days,int64
shipping_mode,string
delivery_status_source,string
late_delivery_risk_source,int64
order_status_source,string
customer_segment,string
market,string


### Observations
- orders_core contains 65,752 records and 18 columns.
- Every order ID is populated and unique.
- All orders matched their validated timestamps.
- Order and shipping timestamps use datetime types.
- The shipping-cancellation indicator uses a Boolean type.
- Cleaned categorical fields use string types.

## 22. Build the Initial Order-Item Analytical Table

In [23]:
item_column_mapping = {
    "Order Item Id": "order_item_id",
    "Order Id": "order_id",
    "Product Card Id": "product_id",
    "Category Id": "category_id",
    "Department Id": "department_id",
    "Product Name": "product_name",
    "Category Name": "category_name",
    "Department Name": "department_name",
    "Order Item Quantity": "quantity",
    "Order Item Product Price": "recorded_unit_price",
    "Sales": "recorded_line_sales",
    "Order Item Discount": "recorded_discount_amount",
    "Order Item Discount Rate": "reported_discount_rate",
    "Order Item Total": "recorded_net_amount",
    "Order Profit Per Order": "recorded_item_profit",
    "Order Item Profit Ratio": "reported_profit_ratio",
}

item_candidates = df[list(item_column_mapping)].copy()

# Carry forward the validated categorical cleaning.
for column in item_column_mapping:
    if column in categorical_clean.columns:
        item_candidates[column] = categorical_clean[column]

order_items_core = item_candidates.rename(columns=item_column_mapping)
order_items_core["product_name"] = (
    order_items_core["product_name"].astype("string")
)

missing_item_ids = order_items_core["order_item_id"].isna()
duplicate_item_ids = order_items_core["order_item_id"].duplicated()
missing_order_ids = order_items_core["order_id"].isna()

orphan_items = (
    order_items_core["order_id"].notna()
    & ~order_items_core["order_id"].isin(orders_core["order_id"])
)

assert len(order_items_core) == len(df)
assert not missing_item_ids.any()
assert not duplicate_item_ids.any()
assert not missing_order_ids.any()
assert not orphan_items.any()

print(f"Item records: {len(order_items_core):,}")
print(f"Columns: {order_items_core.shape[1]}")
print(f"Distinct orders represented: {order_items_core['order_id'].nunique():,}")
print(f"Missing item IDs: {missing_item_ids.sum():,}")
print(f"Duplicate item IDs: {duplicate_item_ids.sum():,}")
print(f"Missing order IDs: {missing_order_ids.sum():,}")
print(f"Items without a matching order: {orphan_items.sum():,}")

print("\nORDER-ITEM TABLE SCHEMA")
display(order_items_core.dtypes.astype(str).to_frame("data_type"))

Item records: 180,519
Columns: 16
Distinct orders represented: 65,752
Missing item IDs: 0
Duplicate item IDs: 0
Missing order IDs: 0
Items without a matching order: 0

ORDER-ITEM TABLE SCHEMA


,data_type
order_item_id,int64
order_id,int64
product_id,int64
category_id,int64
department_id,int64
product_name,string
category_name,string
department_name,string
quantity,int64
recorded_unit_price,float64


## 23A. Check Remaining Source Fields

In [24]:
# 1. Identify empty and constant columns without displaying their values.
field_variation = pd.DataFrame({
    "distinct_non_null_values": df.nunique(dropna=True),
    "missing_rows": df.isna().sum(),
})

print("EMPTY OR CONSTANT SOURCE FIELDS")
display(
    field_variation.loc[
        field_variation["distinct_non_null_values"] <= 1
    ]
)

# 2. Test whether the separate product-price field duplicates item price.
product_price = df["Product Price"]
item_price = df["Order Item Product Price"]

valid_prices = np.isfinite(product_price) & np.isfinite(item_price)

price_matches = pd.Series(
    np.isclose(product_price, item_price, atol=0.01, rtol=0),
    index=df.index
)

print("\nPRODUCT PRICE VS ITEM PRICE")
display(pd.DataFrame([{
    "comparable_rows": int(valid_prices.sum()),
    "missing_or_non_finite_rows": int((~valid_prices).sum()),
    "exact_matches": int(
        (valid_prices & product_price.eq(item_price)).sum()
    ),
    "outside_0.01_tolerance": int(
        (valid_prices & ~price_matches).sum()
    ),
    "max_absolute_difference": (
        product_price - item_price
    ).abs().loc[valid_prices].max(),
}]))

# 3. Check numerical coordinate bounds without displaying locations.
coordinate_results = []

for column, lower, upper in [
    ("Latitude", -90, 90),
    ("Longitude", -180, 180),
]:
    values = df[column]
    valid = np.isfinite(values)

    coordinate_results.append({
        "field": column,
        "missing_or_non_finite_rows": int((~valid).sum()),
        "outside_valid_bounds": int(
            (valid & ~values.between(lower, upper)).sum()
        ),
    })

print("\nCOORDINATE BOUNDS")
display(pd.DataFrame(coordinate_results))

EMPTY OR CONSTANT SOURCE FIELDS


,distinct_non_null_values,missing_rows
Customer Email,1,0
Customer Password,1,0
Product Description,0,180519
Product Status,1,0



PRODUCT PRICE VS ITEM PRICE


,comparable_rows,missing_or_non_finite_rows,exact_matches,outside_0.01_tolerance,max_absolute_difference
0,180519,0,180519,0,0.0



COORDINATE BOUNDS


,field,missing_or_non_finite_rows,outside_valid_bounds
0,Latitude,0,0
1,Longitude,0,0


### Observations
- Customer Email, Customer Password, and Product Status each contain
  one distinct populated value.
- Product Description is entirely missing.
- Product Price exactly equals Order Item Product Price on all
  180,519 records.
- Latitude and Longitude have no missing, non-finite, or
  out-of-bounds values.

## 23B. Screen Item-Level Numerical Outliers

In [25]:
outlier_fields = [
    "quantity",
    "recorded_unit_price",
    "recorded_line_sales",
    "recorded_discount_amount",
    "recorded_net_amount",
    "recorded_item_profit",
]

outlier_flags = pd.DataFrame(index=order_items_core.index)
outlier_results = []

for field in outlier_fields:
    values = order_items_core[field]
    valid = np.isfinite(values)
    q1, q3 = values.loc[valid].quantile([0.25, 0.75])
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    below = valid & values.lt(lower_fence)
    above = valid & values.gt(upper_fence)

    outlier_flags[field] = below | above

    outlier_results.append({
        "field": field,
        "iqr": iqr,
        "lower_fence": lower_fence,
        "upper_fence": upper_fence,
        "below_fence": int(below.sum()),
        "above_fence": int(above.sum()),
        "flagged_items": int((below | above).sum()),
        "non_finite_items": int((~valid).sum()),
    })

outlier_summary = pd.DataFrame(outlier_results)

# Store flags with the business identifier for later review.
item_outlier_flags = pd.concat(
    [
        order_items_core[["order_item_id"]],
        outlier_flags.add_prefix("iqr_flag_"),
    ],
    axis=1
)

item_outlier_flags["any_iqr_flag"] = outlier_flags.any(axis=1)

print("ITEM-LEVEL IQR SCREEN")
display(outlier_summary)

print(
    "\nItems flagged in at least one measure:",
    f"{item_outlier_flags['any_iqr_flag'].sum():,}"
)
print(f"Core item records retained: {len(order_items_core):,}")

ITEM-LEVEL IQR SCREEN


,field,iqr,lower_fence,upper_fence,below_fence,above_fence,flagged_items,non_finite_items
0,quantity,2.000000,-2.000000,6.000000,0,0,0,0
1,recorded_unit_price,149.990005,-174.985008,424.975014,0,2048,2048,0
2,recorded_line_sales,179.970009,-149.975010,569.905025,0,488,488,0
3,recorded_discount_amount,24.590000,-31.484999,66.874999,0,7537,7537,0
4,recorded_net_amount,143.019997,-110.149998,461.929989,0,1943,1943,0
5,recorded_item_profit,57.800003,-79.700005,151.500008,13727,5215,18942,0



Items flagged in at least one measure: 25,789
Core item records retained: 180,519


### Observations
- Profit flags include 13,727 values below the lower fence and
  5,215 above the upper fence.
- Overall, 25,789 distinct items have at least one flag.
- No non-finite values were found in the six screened measures.
- All 180,519 core item records were retained.

## 24. Consolidate the Data Quality Summary

In [26]:
quality_rows = []

def add_quality_check(check, count, unit, severity, decision):
    quality_rows.append({
        "check": check,
        "affected_count": int(count),
        "unit": unit,
        "status": "PASS" if count == 0 else severity,
        "decision": decision,
    })


# Structural issues would block use of the core tables.
structural_checks = [
    ("Missing order IDs",
     orders_core["order_id"].isna().sum(), "orders"),
    ("Duplicate order IDs",
     orders_core["order_id"].duplicated().sum(), "orders"),
    ("Missing item IDs",
     order_items_core["order_item_id"].isna().sum(), "items"),
    ("Duplicate item IDs",
     order_items_core["order_item_id"].duplicated().sum(), "items"),
    ("Missing item-to-order IDs",
     order_items_core["order_id"].isna().sum(), "items"),
    ("Items referencing absent orders",
     (~order_items_core["order_id"].isin(orders_core["order_id"])).sum(),
     "items"),
]

for check, count, unit in structural_checks:
    add_quality_check(
        check, count, unit, "FAIL",
        "Resolve any affected records before downstream loading."
    )

add_quality_check(
    "Missing core timestamps",
    orders_core[["order_datetime", "shipping_datetime"]].isna().sum().sum(),
    "timestamp values", "FAIL", "Require populated validated timestamps."
)

add_quality_check(
    "Shipping timestamp before order timestamp",
    (orders_core["shipping_datetime"] < orders_core["order_datetime"]).sum(),
    "orders", "FAIL", "Investigate chronology violations."
)

add_quality_check(
    "Calendar-day reconciliation mismatch",
    duration_diagnostics["calendar_days"].ne(
        duration_diagnostics["Days for shipping (real)"]
    ).sum(),
    "orders", "FAIL", "Investigate deviations from the observed convention."
)

# Source limitations remain documented even when fields are excluded.
for field in df.columns[df.isna().any()]:
    add_quality_check(
        f"Source missingness: {field}",
        df[field].isna().sum(),
        "source item rows", "WARN",
        "Excluded from current core tables; preserve source missingness."
    )

undocumented = set(dataset_names) - set(dictionary_keys)

add_quality_check(
    "Source fields without dictionary definitions",
    len(undocumented),
    "fields", "WARN", "Document the gap; do not invent a definition."
)

# Reuse the completed financial checks.
for result in commercial_validation_report.to_dict("records"):
    add_quality_check(
        result["relationship"],
        result["outside_tolerance"],
        "items", "WARN",
        "Screening tolerance 0.01; preserve source values and investigate differences."
    )

add_quality_check(
    "At least one item-level IQR flag",
    item_outlier_flags["any_iqr_flag"].sum(),
    "items", "INFO", "Retain records; investigate relevant patterns during EDA."
)

quality_summary = pd.DataFrame(quality_rows)

print("DATA QUALITY SUMMARY")
display(
    quality_summary[
        ["check", "affected_count", "unit", "status"]
    ]
)

print("\nCHECK COUNTS BY STATUS")
display(
    quality_summary.groupby("status")
    .size()
    .rename("check_count")
    .to_frame()
)

DATA QUALITY SUMMARY


,check,affected_count,unit,status
0,Missing order IDs,0,orders,PASS
1,Duplicate order IDs,0,orders,PASS
2,Missing item IDs,0,items,PASS
3,Duplicate item IDs,0,items,PASS
4,Missing item-to-order IDs,0,items,PASS
5,Items referencing absent orders,0,items,PASS
6,Missing core timestamps,0,timestamp values,PASS
7,Shipping timestamp before order timestamp,0,orders,PASS
8,Calendar-day reconciliation mismatch,0,orders,PASS
9,Source missingness: Customer Lname,8,source item rows,WARN



CHECK COUNTS BY STATUS


,check_count
status,
INFO,1
PASS,12
WARN,8


### Observations
- The quality summary contains 21 checks:
  - 12 PASS.
  - 8 WARN.
  - 1 INFO.
  - 0 FAIL.
- Warnings cover source missingness, an undocumented field,
  and three financial arithmetic relationships.
- The informational finding records the item-level IQR flags.

## 25. Document Core Fields and Source-to-Target Transformations

In [27]:
# Preserve the original source descriptions separately.
source_definitions = {
    normalise_field_name(field): str(description).lstrip(":").strip()
    for field, description in data_dictionary[
        ["FIELDS", "DESCRIPTION"]
    ].itertuples(index=False, name=None)
}

# Clarify fields whose source names or descriptions are misleading.
definition_overrides = {
    "actual_shipping_calendar_days":
        "Recorded shipping duration; matches calendar-date difference in this file.",
    "late_delivery_risk_source":
        "Source binary late-shipping label; cancellations are labelled zero.",
    "quantity":
        "Product units on this order-item record.",
    "recorded_unit_price":
        "Recorded item unit price before discount; currency unverified.",
    "recorded_line_sales":
        "Recorded item sales; matches quantity times unit price within tolerance.",
    "recorded_discount_amount":
        "Recorded discount for this item; preserve despite rate-formula warnings.",
    "reported_discount_rate":
        "Published fractional discount rate; original calculation precision unverified.",
    "recorded_net_amount":
        "Recorded net item amount; small sales-minus-discount residuals documented.",
    "recorded_item_profit":
        "Recorded item-level profit measure; accounting definition unverified.",
    "reported_profit_ratio":
        "Published item profit ratio; original calculation precision unverified.",
}

# Fields added outside the direct rename mappings.
special_fields = {
    "order_datetime": (
        ["order date (DateOrders)"],
        "Parse using %m/%d/%Y %H:%M; consolidate by Order Id.",
        "Parsed order timestamp; source timezone unspecified."
    ),
    "shipping_datetime": (
        ["shipping date (DateOrders)"],
        "Parse using %m/%d/%Y %H:%M; consolidate by Order Id.",
        "Parsed shipment timestamp, not confirmed customer receipt time."
    ),
    "is_shipping_cancelled": (
        ["Delivery Status"],
        "Trim text; compare with 'Shipping canceled'.",
        "True when the source shipping outcome is cancelled."
    ),
    "shipping_days_difference": (
        ["Days for shipping (real)", "Days for shipment (scheduled)"],
        "Subtract scheduled days from recorded actual calendar days.",
        "Signed source-duration difference; retain cancellation separately."
    ),
}

metadata_rows = []

for table_name, table, mapping, grain in [
    ("orders_core", orders_core, order_column_mapping, "One row per order"),
    ("order_items_core", order_items_core,
     item_column_mapping, "One row per order item"),
]:
    inverse_mapping = {target: source for source, target in mapping.items()}

    for column in table.columns:
        if column in special_fields:
            sources, transformation, definition = special_fields[column]
            definition_basis = "Profiling-derived interpretation"
        else:
            source = inverse_mapping[column]
            sources = [source]
            transformation = "Select and rename; preserve recorded value."

            if source in categorical_clean.columns:
                transformation = "Trim surrounding whitespace; rename."
            elif column == "product_name":
                transformation = "Cast to string; rename."

            if table_name == "orders_core":
                transformation += " Consolidate validated order attributes."

            definition = definition_overrides.get(
                column,
                source_definitions.get(normalise_field_name(source), "")
            )
            definition_basis = (
                "Profiling interpretation"
                if column in definition_overrides
                else "Source dictionary"
            )

        metadata_rows.append({
            "table": table_name,
            "column": column,
            "grain": grain,
            "data_type": str(table[column].dtype),
            "null_count": int(table[column].isna().sum()),
            "source_fields": " | ".join(sources),
            "source_description": " | ".join(
                source_definitions.get(normalise_field_name(s), "")
                for s in sources
            ),
            "definition": definition,
            "definition_basis": definition_basis,
            "transformation": transformation,
        })

core_data_dictionary = pd.DataFrame(metadata_rows)

source_to_target_mapping = core_data_dictionary[
    ["table", "column", "source_fields", "transformation"]
].copy()

assert len(core_data_dictionary) == (
    orders_core.shape[1] + order_items_core.shape[1]
)
assert not core_data_dictionary.duplicated(["table", "column"]).any()
assert core_data_dictionary["definition"].str.strip().ne("").all()

print("DOCUMENTED COLUMNS BY TABLE")
display(
    core_data_dictionary.groupby("table")
    .size()
    .rename("documented_columns")
    .to_frame()
)

print(f"\nTotal mapping entries: {len(source_to_target_mapping)}")
print(
    "Fields with null values:",
    int(core_data_dictionary["null_count"].gt(0).sum())
)

print("\nEXAMPLE DOCUMENTED FIELDS")
display(
    core_data_dictionary.loc[
        core_data_dictionary["column"].isin([
            "actual_shipping_calendar_days",
            "is_shipping_cancelled",
            "recorded_line_sales",
            "recorded_item_profit",
        ]),
        ["table", "column", "definition", "definition_basis"]
    ]
)

DOCUMENTED COLUMNS BY TABLE


,documented_columns
table,
order_items_core,16
orders_core,18



Total mapping entries: 34
Fields with null values: 0

EXAMPLE DOCUMENTED FIELDS


,table,column,definition,definition_basis
2,orders_core,actual_shipping_calendar_days,Recorded shipping duration; matches calendar-d...,Profiling interpretation
16,orders_core,is_shipping_cancelled,True when the source shipping outcome is cance...,Profiling-derived interpretation
28,order_items_core,recorded_line_sales,Recorded item sales; matches quantity times un...,Profiling interpretation
32,order_items_core,recorded_item_profit,Recorded item-level profit measure; accounting...,Profiling interpretation


### Observations
- All 18 orders_core columns are documented.
- All 16 order_items_core columns are documented.
- The source-to-target mapping contains 34 entries.
- Neither core table contains null values.
- Source descriptions and profiling interpretations are recorded separately.

## 26A. Save Core Tables and Profiling Reports

In [28]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROFILING_REPORT_DIR = PROJECT_ROOT / "reports" / "data_profiling"
DICTIONARY_DIR = PROJECT_ROOT / "docs" / "data_dictionary"

# Structural failures must be resolved before exporting.
assert not quality_summary["status"].eq("FAIL").any(), (
    "Resolve FAIL checks before saving analytical outputs."
)

for directory in [
    PROCESSED_DIR,
    PROFILING_REPORT_DIR,
    DICTIONARY_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

saved_files = []

# Retain this mapping for the read-back verification cell.
parquet_exports = {
    "orders_core": orders_core,
    "order_items_core": order_items_core,
    "item_outlier_flags": item_outlier_flags,
}

for name, table in parquet_exports.items():
    path = PROCESSED_DIR / f"{name}.parquet"

    table.to_parquet(
        path,
        engine="pyarrow",
        compression="snappy",
        index=False
    )

    saved_files.append({
        "file": str(path.relative_to(PROJECT_ROOT)),
        "rows": len(table),
        "size_bytes": path.stat().st_size,
    })

csv_exports = {
    PROFILING_REPORT_DIR / "data_quality_summary.csv":
        quality_summary,

    PROFILING_REPORT_DIR / "commercial_validation.csv":
        commercial_validation_report,

    PROFILING_REPORT_DIR / "outlier_summary.csv":
        outlier_summary,

    PROFILING_REPORT_DIR / "source_missing_values.csv":
        missing_report.rename_axis("source_field").reset_index(),

    PROFILING_REPORT_DIR / "source_field_variation.csv":
        field_variation.rename_axis("source_field").reset_index(),

    PROFILING_REPORT_DIR / "categorical_trim_audit.csv":
        pd.DataFrame(trim_results),

    PROFILING_REPORT_DIR / "identifier_agreement.csv":
        pd.DataFrame(identifier_results),

    PROFILING_REPORT_DIR / "product_hierarchy_checks.csv":
        pd.DataFrame(hierarchy_results),

    PROFILING_REPORT_DIR / "coordinate_bounds.csv":
        pd.DataFrame(coordinate_results),

    DICTIONARY_DIR / "core_data_dictionary.csv":
        core_data_dictionary,

    DICTIONARY_DIR / "source_to_target_mapping.csv":
        source_to_target_mapping,
}

for path, table in csv_exports.items():
    table.to_csv(path, index=False, encoding="utf-8-sig")

    saved_files.append({
        "file": str(path.relative_to(PROJECT_ROOT)),
        "rows": len(table),
        "size_bytes": path.stat().st_size,
    })

export_manifest = pd.DataFrame(saved_files)

print("Project root:", PROJECT_ROOT)
print(f"Files written: {len(export_manifest)}")
display(export_manifest)

Project root: c:\Users\saths\Desktop\MY Pro\shipsense-360
Files written: 14


,file,rows,size_bytes
0,data\processed\orders_core.parquet,65752,2237116
1,data\processed\order_items_core.parquet,180519,3394352
2,data\processed\item_outlier_flags.parquet,180519,1123215
3,reports\data_profiling\data_quality_summary.csv,21,2369
4,reports\data_profiling\commercial_validation.csv,6,556
5,reports\data_profiling\outlier_summary.csv,6,552
6,reports\data_profiling\source_missing_values.csv,4,155
7,reports\data_profiling\source_field_variation.csv,53,1286
8,reports\data_profiling\categorical_trim_audit.csv,12,391
9,reports\data_profiling\identifier_agreement.csv,3,208


### Observations
- Three Parquet files and eleven CSV files were written.
- The saved core tables contain 65,752 orders and 180,519 items.
- Outlier flags cover 180,519 items.
- All listed files have non-zero sizes.

## 26B. Verify Saved Outputs by Reading Them Back

In [29]:
reloaded_tables = {}
verification_results = []

# 1. Verify Parquet values, data types, columns, and row order.
for name, expected in parquet_exports.items():
    path = PROCESSED_DIR / f"{name}.parquet"
    loaded = pd.read_parquet(path, engine="pyarrow")

    pd.testing.assert_frame_equal(
        loaded,
        expected,
        check_exact=True
    )

    reloaded_tables[name] = loaded

    verification_results.append({
        "check": f"Parquet round trip: {name}",
        "status": "PASS",
    })

# 2. Verify CSV contents using the default float parser.
# CSV infers types when reading, so dtype equality is not required.
# Values, column order, and row order must still match.
for path, expected in csv_exports.items():
    loaded = pd.read_csv(
        path,
        encoding="utf-8-sig",
        keep_default_na=False
    )

    pd.testing.assert_frame_equal(
        loaded,
        expected,
        check_dtype=False,
        check_exact=True
    )

    verification_results.append({
        "check": f"CSV round trip: {path.name}",
        "status": "PASS",
    })

saved_orders = reloaded_tables["orders_core"]
saved_items = reloaded_tables["order_items_core"]
saved_flags = reloaded_tables["item_outlier_flags"]

# 3. Validate saved identifiers and relationships.
assert saved_orders["order_id"].notna().all()
assert saved_orders["order_id"].is_unique

assert saved_items["order_item_id"].notna().all()
assert saved_items["order_item_id"].is_unique
assert saved_items["order_id"].notna().all()

assert saved_items["order_id"].isin(
    saved_orders["order_id"]
).all()

assert saved_flags["order_item_id"].notna().all()
assert saved_flags["order_item_id"].is_unique
assert len(saved_flags) == len(saved_items)

assert saved_flags["order_item_id"].isin(
    saved_items["order_item_id"]
).all()

verification_results.append({
    "check": "Saved keys, order relationships, and flag coverage",
    "status": "PASS",
})

# 4. Reconcile saved quantities and amounts with the source.
saved_measure_mapping = {
    "quantity": "Order Item Quantity",
    "recorded_line_sales": "Sales",
    "recorded_discount_amount": "Order Item Discount",
    "recorded_net_amount": "Order Item Total",
    "recorded_item_profit": "Order Profit Per Order",
}

for target, source in saved_measure_mapping.items():
    assert np.isclose(
        saved_items[target].sum(),
        df[source].sum(),
        atol=0.01,
        rtol=0
    ), f"Saved total does not reconcile: {target}"

    verification_results.append({
        "check": f"Source-to-saved total: {target}",
        "status": "PASS",
    })

# Save verification evidence after every assertion succeeds.
readback_verification = pd.DataFrame(verification_results)

verification_path = (
    PROFILING_REPORT_DIR / "readback_verification.csv"
)

readback_verification.to_csv(
    verification_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Exported files verified:",
    len(parquet_exports) + len(csv_exports)
)
print(f"Orders reloaded: {len(saved_orders):,}")
print(f"Items reloaded: {len(saved_items):,}")
print(f"Verification checks passed: {len(readback_verification)}")

display(readback_verification)

print("\nVerification report saved:", verification_path)

Exported files verified: 14
Orders reloaded: 65,752
Items reloaded: 180,519
Verification checks passed: 20


,check,status
0,Parquet round trip: orders_core,PASS
1,Parquet round trip: order_items_core,PASS
2,Parquet round trip: item_outlier_flags,PASS
3,CSV round trip: data_quality_summary.csv,PASS
4,CSV round trip: commercial_validation.csv,PASS
5,CSV round trip: outlier_summary.csv,PASS
6,CSV round trip: source_missing_values.csv,PASS
7,CSV round trip: source_field_variation.csv,PASS
8,CSV round trip: categorical_trim_audit.csv,PASS
9,CSV round trip: identifier_agreement.csv,PASS



Verification report saved: c:\Users\saths\Desktop\MY Pro\shipsense-360\reports\data_profiling\readback_verification.csv


### Observations
- All 14 exported files passed read-back verification.
- Saved identifiers, order-item relationships, and outlier-flag
  coverage passed validation.
- Quantity and all four financial totals reconcile with the source.
- All 20 verification checks passed.